# Pricing

---

## Basic Pricing and DF

#### Forward rate
- $f$ ( $t_i$ , $t_{i+1}$ ) is the forward rate between $t_i$ and $t_{i+1}$  
- This means if *1* is invested at $t_i$, it grows to *1 + f* ( $t_i$ , $t_{i+1}$ ) $\tau_{(i,i+1)}$
- $DF$($0$, $t$) is discount factor between the period $0$ and $t$
Hence
$$
(1 + f ( t_i , t_{i+1} )\tau_{(i,i+1)}) * DF(t_i, t_{i+1}) = 1  \\
DF(t_i, t_{i+1}) = \frac{1}{1 + f ( t_i , t_{i+1} )\tau_{(i,i+1)}}
$$

Given that
$$
DF(0, t_{i+1}) = DF(0, t_{i}) *  DF(t_i, t_{i+1}) \\
$$
We can say:
$$
DF(0, t_{i+1}) = DF(0, t_{i}) *  \frac{1}{1 + f ( t_i , t_{i+1} )\tau_{(i,i+1)}}
$$
- Why is this valid for overnight compounded SOFR?
- For SOFR, actual floating leg over a period is built by daily compounding of overnight fixings.
- But once the market gives you the equivalent forward rate over the whole accrual period, you can represent that whole period by one $f$

#### Continuous-compounding rate  
If instead the forward rate was continuously compounded
$$
DF(0,t_{i+1}) = DF(0,t_i) * e^{-f(t_i,t_{i+1})\tau_{(i,i+1)}}
$$

#### Standard OIS swap pricing (simplified)

For a standard SOFR OIS, once you know the discount factors at the coupon dates, pricing becomes much simpler because the floating leg can be written directly in terms of those DFs.
Based on the assumtions that:
- It is a standard OIS
- The floating leg is pure compounded SOFR with no spread
- The same SOFR curve is used for projection and discounting

In an OIS/SOFR swap, the floating coupon over one period is:
$$
Coupon_\text{floating} = notional \times (growth  factor  - 1)
$$

Suppose I have $1 at $T_{i-1}$  
$1 paid at $T_{i-1}$ has a today-value of:
$$
1 \times DF(0,T_{i-1})
$$
We want an amount $X$ paid at $T_i$ that also has today-value of  $1$ x $DF(0,T_{t-1})$
$$
X \times DF(0,T_i) = 1 \times DF(0,T_{i-1})
$$
So
$$
X = 1 \times \frac{DF(0,T_{i-1})}{DF(0,T_i)}
$$
Instead of 1, if the notional was $N$ at $T_{i-1}$, the interest earned between $T_{i-1}$ to $T_i$ is:
$$
N \times (\frac{DF(0,T_{i-1})}{DF(0,T_i)} - 1 )
$$
So this is the floating leg coupon. Hence
$$
\begin{aligned}
PV^\text{float}_i
&=
\text{coupon} \times DF(0,T_i)  \\
&=
N \times (\frac{DF(0,T_{i-1})}{DF(0,T_i)} - 1 ) \times DF(0,T_i)  \\
&=
N \times (DF(0,T_{i-1})-DF(0,T_i))
\end{aligned} \\
$$

**Why does the whole leg telescope?**  
Now let's sum all the coupons:
$$
\begin{aligned}
PV^\text{float}_i
&=
N \sum_{i=1}^{n} (DF(0,T_{i-1})-DF(0,T_i))  \\
&= N [DF(0,T_0) - DF(0,T_1) + DF(0,T_1) - DF(0, T_2) + DF(0,T_2) + DF(0,T_3) + .... + DF(0,T_{n-1})+ DF(0,T_n)]  \\
&= N (DF(0,T_0) - DF(0,T_n))
\end{aligned} \\
$$

If the swap starts today then $DF(0, T_0)$ = $1$, so
$$
PV^\text{float} = N (1 - DF(0,T_n))
$$
In this simplified version you do not need to model each future daily SOFR fixing one by one.  
The whole floating leg can be valued from the discount curve alone.
Since the fixed leg is:
$$
PV^\text{fix}_i = N K \sum_{i=1}^{n} \tau_i DF(0,T_i)
$$
At inception
$$
PV^\text{float} = PV^\text{fix}_i
$$
So
$$
N (1 - DF(0,T_n)) = N K \sum_{i=1}^{n} \tau_i DF(0,T_i)
$$

## Swap rate and Annuity

At inception ($t_0$), all linear products (products with no optionality) must have a PV of zero. For our vanilla IRS with notional $N$, this implies that
$$
\begin{aligned}
PV_\text{float}(t_0)
&=
PV_\text{fix}(t_0) \\
&=
N * K \, \underbrace{\sum_{i=1}^{n} \tau_i P(t_0,T_i)}_{A(t_0;T_0,T_n)}
\end{aligned} \\

$$
where $A(t;T_0,T_n)$ denotes the swap annuity, a fixed sum of money paid to someone each year. Rearranging for $K$, we get the fixed rate required to price the swap at par. This quantity is known as the swap rate $S(t;T_0,T_n)$,
$$
S(t_0;T_0,T_n) = 
\frac{PV_{\text{float}}(t_0)/N}{A(t_0;T_0,T_n)}.
$$

Using a given swap rate, we can easily compute the price (MtM) of the corresponding swap at any time $t_0 \leq t \leq T_0$,

$$
\begin{align}
V^{\text{swap}}(t)
&=
N \phi \, \big(S(t;T_0,T_n) A(t;T_0,T_n) - K A(t;T_0,T_n) \big) \\
&= N \phi \, A(t;T_0,T_n) \big(S(t;T_0,T_n)) - K \big).
\end{align}
$$
So the swap value is just annuity * rate difference. This is why people call a swap linear product: the payoff / value depends linearly on $S_t$ - $K$, unlike option which has a kink such as ($S_t$ - $K)^+$

## Risk management

In swap market useage, often:  
PV01 = What happens to PV if I change the swap’s fixed rate by 1 bp? Annuity, coupon sensitivity  
DV01 = What happens to PV if the market curve itself moves by 1 bp? Curve sensitivity

$$
\begin{align}
PV01 &:= \frac{\partial PV^{\text{swap}}}{\partial K} \times 1\text{b.p.}\\
&= N A (t;T_0,T_n) \times 1\text{b.p.}
\end{align}
$$

DV01, for example the change in swap PV for a 1bp move in the SOFR curve is:
$$
\begin{align}
DV01 &:= \frac{\partial PV^{\text{swap}}}{\partial \theta^{\text{SOFR}}} \times 1\text{b.p.} 
\end{align}
$$

A very common practical version is the parallel SOFR-curve DV01:
$$
\begin{align}
DV01 &:=
\frac{\partial PV^{\text{swap}}}{\partial \theta^{\text{SOFR}}} \times 1\text{b.p.}\\
&=
\frac{PV(curve + 0.5bp)-PV(curve - 0.5bp)}{1bp}
\end{align}
$$

If instead they mean bucketed SOFR DV01, then for SOFR pillar $j$,
$$
\begin{align}
DV01 &:=
\frac{\partial PV^{\text{swap}}}{\partial q_j} \times 1\text{b.p.}\\
&=
\frac{PV(q_j + 0.5bp)-PV(q_j - 0.5bp)}{1bp}
\end{align}
$$
Where $q_j$ is the bumped SOFR curve input at a tenor $j$. Depending on the system, $q_j$ may be a zero rate or market quote used in calibration. And bucket sensitivities add up approximately to the parallel DV01.

Before IBOR decommission, a multi-curve IRS carries exposure to:
- Discounting curve risk (OIS DV01)
- Forward/index curve risk (IBOR DV01)
- Basis risk, i.e. sensitivity to the spread between OIS and IBOR curves.

A linear rates desk typically needs to:
- hedge OIS DV01 using OIS swaps,
- hedge IBOR DV01 using vanilla IBOR swaps or FRAs,
- manage basis risk using OIS-IBOR basis swaps, and
- monitor residual risks arising from curve interpolation.

Risks against ColVA (e.g. collateral conventions, CSA discounting switches, collateral optionalities etc) and FVA (funding costs from collateral mismatch) are typically offloaded to the XVA desk. Convexity adjustments, curve interpolation methodology, and XVAs are typically highly model dependent - thus exposed to significant model risks.
